In [1]:
import pandas as pd

In [2]:
views = pd.read_csv(
    '../data/feed-views.log', sep='\t', header=None, names=['datetime', 'user'])

Преобрауем в datetime64:

In [4]:
views['datetime'] = pd.to_datetime(views['datetime'])

In [5]:
views['year'] = views['datetime'].dt.year      
views['month'] = views['datetime'].dt.month  
views['day'] = views['datetime'].dt.day       
views['hour'] = views['datetime'].dt.hour      
views['minute'] = views['datetime'].dt.minute  
views['second'] = views['datetime'].dt.second 

Границы интервалов часов:

In [7]:
hours_intervals = [-1, 3, 6, 10, 16, 19, 23]

Категории времени дня:

In [9]:
time_categories = ['night', 'early morning', 'morning', 'afternoon', 'early evening', 'evening']

In [10]:
views['daytime'] = pd.cut(views['hour'], bins=hours_intervals, labels=time_categories)

устанавливаем userа как индекс

In [12]:
views = views.set_index('user')

Подсчитываем элементы и сортируем:

In [14]:
views.count()

datetime    1076
year        1076
month       1076
day         1076
hour        1076
minute      1076
second      1076
daytime     1076
dtype: int64

In [15]:
views['daytime'].value_counts()

daytime
evening          509
afternoon        252
early evening    145
night            129
morning           36
early morning      5
Name: count, dtype: int64

In [16]:
views['hour'].min()

0

In [17]:
views['hour'].max()

23

In [18]:
max_night_hour = views[views['daytime'] == 'night']['hour'].max()
max_night_hour

3

Минимальный час для утреннего времени и для юзеров утром:

In [20]:
min_morning_hour = views[views['daytime'] == 'morning']['hour'].min()
min_morning_hour

8

In [21]:
views[(views['daytime'] == 'morning') & (views['hour'] == min_morning_hour)].index.tolist()

['alexander', 'alexander']

Мода для часа и мода в течение суток:

In [23]:
views.hour.mode()

0    22
Name: hour, dtype: int32

In [24]:
views['daytime'].mode()[0]

'evening'

3 самых ранних утром и 3 самых поздних вечером:

In [26]:
views[views['daytime'] == 'morning'].nsmallest(3, 'hour')[['hour', 'datetime']]

,hour,datetime
user,,
alexander,8,2020-05-15 08:16:03.918402
alexander,8,2020-05-15 08:35:01.471463
artem,9,2020-04-24 09:42:47.598208


In [27]:
views[views['daytime'] == 'evening'].nlargest(3, 'hour')[['hour', 'datetime']]

,hour,datetime
user,,
konstantin,23,2020-04-18 23:06:34.198911
artem,23,2020-04-18 23:40:32.666884
artem,23,2020-04-19 23:10:03.761243


In [28]:
stats = views[['hour', 'minute', 'second']].describe()
stats

,hour,minute,second
count,1076.000000,1076.000000,1076.000000
mean,16.249071,29.629182,29.500929
std,6.955490,17.689388,17.405506
min,0.000000,0.000000,0.000000
25%,13.000000,14.000000,14.000000
50%,19.000000,29.000000,30.000000
75%,22.000000,46.000000,45.000000
max,23.000000,59.000000,59.000000


Межквартильный размах:

In [30]:
Q1 = stats.loc['25%', 'hour']
Q3 = stats.loc['75%', 'hour']
iqr = Q3 - Q1
iqr

9.0